# Preprocessing — Band Stacking, Patches, and dNBR Labels

**Environment**: Local CPU

**Part 1**: Band stacking, SCL cloud masking, patch extraction (256x256)
**Part 2**: dNBR label generation from pre/post-fire Sentinel-2 pairs

## 1. Imports y paths

In [ ]:
import sys, os
from pathlib import Path

# Fix: PostgreSQL/PostGIS instala su propio PROJ que entra en conflicto con conda.
# Forzar PROJ_DATA al entorno conda ANTES de importar rasterio o geopandas.
_conda_prefix = Path(sys.executable).parent.parent
_proj_data = _conda_prefix / 'Library' / 'share' / 'proj'
if _proj_data.exists():
    os.environ['PROJ_DATA'] = str(_proj_data)
    os.environ['PROJ_LIB']  = str(_proj_data)
    print(f'PROJ_DATA → {_proj_data}')
else:
    print(f'WARNING: PROJ_DATA not found at {_proj_data}')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.warp
import rasterio.features
import rasterio.windows
from rasterio.crs import CRS
from pathlib import Path
from tqdm import tqdm

RAW_DIR   = Path('..') / 'data' / 'sentinel2' / 'raw'
PROC_DIR  = Path('..') / 'data' / 'sentinel2' / 'processed'
FIRMS_DIR = Path('..') / 'data' / 'firms'
PATCH_DIR = Path('..') / 'data' / 'patches'

for d in [PROC_DIR, PATCH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CRS_UTM = CRS.from_epsg(32721)  # UTM Zone 21S — correcto para Corrientes

BANDS          = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']
PATCH_SIZE     = 256
MAX_CLOUD_FRAC = 0.2
FIRMS_BUFFER_M = 375

print(f'rasterio {rasterio.__version__}')
print(f'CRS: {CRS_UTM}')
print(f'Raw dir: {RAW_DIR.resolve()}')

## 2. Descubrir tiles descargadas

In [ ]:
# Estructura: raw/<item_id>/<date>/B02.jp2
scenes = []
for item_dir in sorted(RAW_DIR.iterdir()):
    for date_dir in sorted(item_dir.iterdir()):
        jp2_files = list(date_dir.glob('*.jp2'))
        if jp2_files:
            scenes.append({
                'item_id': item_dir.name,
                'date': date_dir.name,
                'dir': date_dir,
                'n_bands': len(jp2_files),
            })

print(f'Scenes encontradas: {len(scenes)}')
for s in scenes:
    print(f"  {s['date']}  {s['item_id'][-20:]}  ({s['n_bands']} bandas)")

## 3. Función: procesar una escena

Para cada escena:
- Reproyecta cada banda JP2 a EPSG:32721 a 10 m
- Aplica SCL mask (valores válidos: 4=vegetación, 5=no vegetado, 6=agua)
- Calcula NDVI, NBR, NDWI
- Guarda GeoTIFF multi-banda: [B02, B03, B04, B08, B8A, B11, B12, NDVI, NBR, NDWI, MASK]

In [ ]:
# SCL valores claros (no enmascarar)
SCL_CLEAR = {4, 5, 6}  # vegetation, not_vegetated, water

def reproject_band(src_path, dst_crs, resolution=10):
    """Lee un JP2, reproyecta a dst_crs con resolución dada. Devuelve (data, transform, nodata)."""
    with rasterio.open(src_path) as src:
        # Calcular transform y shape destino
        transform, width, height = rasterio.warp.calculate_default_transform(
            src.crs if src.crs else CRS.from_epsg(32721),
            dst_crs, src.width, src.height, *src.bounds,
            resolution=resolution
        )
        kwargs = src.meta.copy()
        kwargs.update(crs=dst_crs, transform=transform,
                      width=width, height=height, nodata=0)

        data = np.zeros((src.count, height, width), dtype=src.dtypes[0])
        rasterio.warp.reproject(
            source=rasterio.band(src, 1),
            destination=data[0],
            src_transform=src.transform,
            src_crs=src.crs if src.crs else CRS.from_epsg(32721),
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=rasterio.warp.Resampling.bilinear
        )
    return data[0], transform, kwargs


def compute_index(a, b):
    """(a - b) / (a + b) con protección contra división por cero."""
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    denom = a + b
    with np.errstate(invalid='ignore', divide='ignore'):
        idx = np.where(denom != 0, (a - b) / denom, 0.0)
    return idx.astype(np.float32)


def process_scene(scene, out_dir, crs=CRS_UTM, resolution=10):
    """
    Procesa una escena S2: reproyecta, enmascara, calcula índices.
    Guarda GeoTIFF con 11 bandas: B02 B03 B04 B08 B8A B11 B12 NDVI NBR NDWI MASK
    """
    scene_dir = scene['dir']
    date = scene['date']
    item_id = scene['item_id']

    out_path = out_dir / f"{item_id[-15:]}_{date}.tif"  # nombre corto
    if out_path.exists():
        print(f'  SKIP {out_path.name} (ya existe)')
        return out_path

    # --- SCL mask (20m → reproyectar a 10m) ---
    scl_path = scene_dir / 'SCL.jp2'
    scl_data, scl_transform, scl_meta = reproject_band(scl_path, crs, resolution)
    clear_mask = np.isin(scl_data, list(SCL_CLEAR))  # True = pixel válido

    shape = scl_data.shape  # (H, W)
    transform = scl_transform

    # --- Leer y reproyectar bandas espectrales ---
    band_data = {}
    for band in BANDS:
        jp2 = scene_dir / f'{band}.jp2'
        data, _, _ = reproject_band(jp2, crs, resolution)
        # Redimensionar si es necesario (SCL a 20m puede diferir ligeramente)
        if data.shape != shape:
            from rasterio.enums import Resampling as Rs
            data = rasterio.warp.reproject(
                data, np.empty(shape, dtype=data.dtype),
                src_transform=_, src_crs=crs,
                dst_transform=transform, dst_crs=crs,
                resampling=Rs.bilinear
            )[0]
        band_data[band] = data

    # --- Índices ---
    B08 = band_data['B08'].astype(np.float32)
    NDVI = compute_index(B08, band_data['B04'])
    NBR  = compute_index(B08, band_data['B12'])
    NDWI = compute_index(band_data['B03'], B08)

    # --- Apilar 11 bandas ---
    stack = np.stack([
        band_data['B02'], band_data['B03'], band_data['B04'],
        band_data['B08'], band_data['B8A'], band_data['B11'], band_data['B12'],
        (NDVI * 10000).astype(np.int16),
        (NBR  * 10000).astype(np.int16),
        (NDWI * 10000).astype(np.int16),
        clear_mask.astype(np.uint8),  # 1=válido, 0=nube/sombra
    ], axis=0)

    profile = {
        'driver': 'GTiff', 'dtype': 'int16', 'compress': 'lzw',
        'crs': crs, 'transform': transform,
        'width': shape[1], 'height': shape[0],
        'count': stack.shape[0], 'nodata': -9999,
    }

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(stack)
        dst.update_tags(
            band_names='B02,B03,B04,B08,B8A,B11,B12,NDVI,NBR,NDWI,MASK',
            date=date, item_id=item_id,
        )

    size_mb = out_path.stat().st_size / 1e6
    pct_clear = clear_mask.mean() * 100
    print(f'  OK   {out_path.name}  ({size_mb:.0f} MB, {pct_clear:.1f}% clear)')
    return out_path


print('Funciones definidas.')

## 4. Procesar todas las escenas

In [ ]:
processed_paths = []

for scene in scenes:
    print(f"\nProcesando {scene['date']} — {scene['item_id'][-20:]}")
    path = process_scene(scene, PROC_DIR)
    processed_paths.append(path)

print(f'\n{len(processed_paths)} escenas procesadas.')

## 5. Rasterizar FIRMS → fire mask

For each scene, generates a binary mask (0/1) with VIIRS detections
en un rango de ±2 días alrededor de la fecha de adquisición.

In [ ]:
# Cargar CSV FIRMS
firms_csv = FIRMS_DIR / 'viirs_snpp_corrientes_2021-12_2022-02.csv'
df_firms = pd.read_csv(firms_csv, parse_dates=['acq_date'])
print(f'FIRMS: {len(df_firms)} detecciones cargadas')

# Crear GeoDataFrame en WGS84, luego reproyectar a UTM 21S
gdf_firms = gpd.GeoDataFrame(
    df_firms,
    geometry=gpd.points_from_xy(df_firms['longitude'], df_firms['latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:32721')

# Buffer de 375 m (resolución VIIRS)
gdf_firms['geometry'] = gdf_firms.geometry.buffer(FIRMS_BUFFER_M)
print(f'Buffer {FIRMS_BUFFER_M} m aplicado.')


def rasterize_fire_mask(scene_date_str, processed_tif, gdf_fire, window_days=2):
    """
    Rasteriza detecciones FIRMS dentro de ±window_days de scene_date_str
    sobre la grilla del GeoTIFF procesado. Guarda como <tif_stem>_firemask.tif
    """
    scene_date = pd.Timestamp(scene_date_str[:8])  # YYYYMMDD
    t0 = scene_date - pd.Timedelta(days=window_days)
    t1 = scene_date + pd.Timedelta(days=window_days)
    subset = gdf_fire[(gdf_fire['acq_date'] >= t0) & (gdf_fire['acq_date'] <= t1)]

    out_path = processed_tif.parent / (processed_tif.stem + '_firemask.tif')

    with rasterio.open(processed_tif) as src:
        meta = src.meta.copy()
        meta.update(count=1, dtype='uint8', nodata=255, compress='lzw')
        fire_mask = np.zeros((src.height, src.width), dtype=np.uint8)

        if len(subset) > 0:
            shapes = [(geom, 1) for geom in subset.geometry if geom is not None]
            if shapes:
                rasterio.features.rasterize(
                    shapes, out=fire_mask,
                    transform=src.transform,
                    fill=0, dtype='uint8'
                )

    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(fire_mask[np.newaxis, ...])
        dst.update_tags(scene_date=scene_date_str, window_days=str(window_days),
                        n_detections=str(len(subset)))

    pct_fire = fire_mask.mean() * 100
    print(f'  {scene_date_str[:8]}  {len(subset):>5} dets → {pct_fire:.2f}% fire pixels')
    return out_path


print('\nRasterizando FIRMS:')
fire_mask_paths = []
for scene, proc_path in zip(scenes, processed_paths):
    fmask = rasterize_fire_mask(scene['date'], proc_path, gdf_firms)
    fire_mask_paths.append(fmask)

## 6. Visualización — una escena procesada

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Usar la escena peak_fire (índice 2 o 3)
idx = 2
proc_tif  = processed_paths[idx]
fire_tif  = fire_mask_paths[idx]
scene_date = scenes[idx]['date']

with rasterio.open(proc_tif) as src:
    # RGB: B04, B03, B02 (bandas 3, 2, 1 — índice 0-based: 2, 1, 0)
    r = src.read(3).astype(np.float32)  # B04
    g = src.read(2).astype(np.float32)  # B03
    b = src.read(1).astype(np.float32)  # B02
    ndvi = src.read(8).astype(np.float32) / 10000
    mask = src.read(11)  # valid mask

with rasterio.open(fire_tif) as src:
    fire = src.read(1)

# Normalizar RGB
def norm(x, p_low=2, p_high=98):
    lo, hi = np.percentile(x[x > 0], [p_low, p_high]) if (x > 0).any() else (0, 1)
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

rgb = np.dstack([norm(r), norm(g), norm(b)])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(rgb)
axes[0].set_title(f'RGB (B04/B03/B02) — {scene_date}')
axes[0].axis('off')

im1 = axes[1].imshow(ndvi, cmap='RdYlGn', vmin=-0.3, vmax=0.8)
axes[1].set_title('NDVI')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], shrink=0.7)

# Overlay fire mask sobre RGB
axes[2].imshow(rgb)
fire_overlay = np.zeros((*fire.shape, 4), dtype=np.float32)
fire_overlay[fire == 1] = [1, 0, 0, 0.6]  # rojo semitransparente
axes[2].imshow(fire_overlay)
axes[2].set_title('RGB + Fire mask (FIRMS ±2 días)')
axes[2].axis('off')
patch = mpatches.Patch(color='red', alpha=0.6, label='Fire detection')
axes[2].legend(handles=[patch], loc='lower right')

plt.tight_layout()
out = Path('..') / 'results' / f'preprocess_check_{scene_date}.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado en {out}')

## 7. Generación de patches 256×256

Sliding window sobre cada escena procesada. Guarda pares (imagen, fire_mask) como archivos `.npy`.
Solo conserva patches con < 20% de píxeles enmascarados.

In [ ]:
STRIDE = PATCH_SIZE  # sin solapamiento; usar PATCH_SIZE//2 para data augmentation

(PATCH_DIR / 'images').mkdir(exist_ok=True)
(PATCH_DIR / 'masks').mkdir(exist_ok=True)

total_patches = 0
kept_patches  = 0

for scene, proc_tif, fire_tif in zip(scenes, processed_paths, fire_mask_paths):
    date = scene['date']
    prefix = proc_tif.stem
    scene_patches = 0
    scene_kept = 0

    with rasterio.open(proc_tif) as src_img, rasterio.open(fire_tif) as src_mask:
        H, W = src_img.height, src_img.width

        for row in range(0, H - PATCH_SIZE + 1, STRIDE):
            for col in range(0, W - PATCH_SIZE + 1, STRIDE):
                window = rasterio.windows.Window(col, row, PATCH_SIZE, PATCH_SIZE)

                img_patch  = src_img.read(window=window)   # (11, 256, 256)
                mask_patch = src_mask.read(1, window=window)  # (256, 256)

                # La banda 11 (índice 10) es el valid mask
                valid = img_patch[10]
                cloud_frac = 1 - valid.mean()

                total_patches += 1
                scene_patches += 1

                if cloud_frac > MAX_CLOUD_FRAC:
                    continue  # demasiadas nubes

                patch_id = f'{prefix}_r{row:05d}_c{col:05d}'
                np.save(PATCH_DIR / 'images' / f'{patch_id}.npy', img_patch)
                np.save(PATCH_DIR / 'masks'  / f'{patch_id}.npy', mask_patch)

                kept_patches += 1
                scene_kept   += 1

    pct = scene_kept / scene_patches * 100 if scene_patches > 0 else 0
    print(f'  {date}: {scene_kept}/{scene_patches} patches conservados ({pct:.0f}%)')

print(f'\nTotal: {kept_patches}/{total_patches} patches ({kept_patches/total_patches*100:.1f}%)')
print(f'Guardados en {PATCH_DIR}')

## 8. Verificación final

In [ ]:
img_files  = sorted((PATCH_DIR / 'images').glob('*.npy'))
mask_files = sorted((PATCH_DIR / 'masks').glob('*.npy'))

print(f'Patches imagen: {len(img_files)}')
print(f'Patches máscara: {len(mask_files)}')

# Inspeccionar uno
if img_files:
    sample = np.load(img_files[len(img_files)//2])
    smask  = np.load(mask_files[len(mask_files)//2])
    print(f'\nSample imagen: shape={sample.shape}, dtype={sample.dtype}')
    print(f'  B02 range: [{sample[0].min()}, {sample[0].max()}]')
    print(f'  NDVI range: [{sample[7].min()/10000:.3f}, {sample[7].max()/10000:.3f}]')
    print(f'Sample máscara: shape={smask.shape}, valores únicos={np.unique(smask)}')
    fire_patches = sum(1 for f in mask_files if np.load(f).max() > 0)
    print(f'\nPatches con fuego activo: {fire_patches}/{len(mask_files)} ({fire_patches/len(mask_files)*100:.1f}%)')

---

## Part 2 — dNBR Label Generation

In [ ]:
import sys, os
from pathlib import Path

_conda_prefix = Path(sys.executable).parent.parent
_proj_data = _conda_prefix / 'Library' / 'share' / 'proj'
if _proj_data.exists():
    os.environ['PROJ_DATA'] = str(_proj_data)
    os.environ['PROJ_LIB']  = str(_proj_data)

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PATCH_DIR    = Path('..') / 'data' / 'patches'
IMG_DIR      = PATCH_DIR / 'images'
MASK_FIRMS   = PATCH_DIR / 'masks'
MASK_DNBR    = PATCH_DIR / 'masks_dnbr'
MASK_DNBR.mkdir(exist_ok=True)

# Índice 0-based en el stack de 11 bandas: [B02,B03,B04,B08,B8A,B11,B12,NDVI,NBR,NDWI,MASK]
NBR_IDX  = 8
MASK_IDX = 10   # 1=píxel válido, 0=nube

print(f'Patches en IMG_DIR: {len(list(IMG_DIR.glob("*.npy")))}')
print(f'Output: {MASK_DNBR.resolve()}')

In [ ]:
# ── Scene classification ───────────────────────────────────────────────────
#
# Stem del patch = stem del GeoTIFF procesado = "{item_id[-15:]}_{date}"
#
# Tile 21JVJ — true dNBR pair
PRE_21JVJ  = '20221230T031220_20211229'
POST_21JVJ = '20240507T114228_20220128'

# Tile 21JWK — true dNBR pair
PRE_21JWK  = '20221228T160256_20211231'
POST_21JWK = '20240516T100304_20220219'

# Map: post scene → pre scene (to recover pre-fire NBR)
DNBR_PAIRS = {
    POST_21JVJ: PRE_21JVJ,
    POST_21JWK: PRE_21JWK,
}

# Pre-fire scenes → all-zeros mask (no scar yet)
PRE_FIRE_STEMS = {PRE_21JVJ, PRE_21JWK}

# Umbrales
DNBR_THRESHOLD = 0.10   # dNBR > 0.10 → cicatriz (baja a alta severidad)
NBR_THRESHOLD  = -0.05  # NBR < -0.05 → píxel quemado (imágenes sin par)

print('Escenas configuradas:')
print(f'  Pre-fuego (zeros): {PRE_FIRE_STEMS}')
print(f'  dNBR verdadero:   {list(DNBR_PAIRS.keys())}')
print(f'  NBR umbral:       21JUH (20220131) + 21JVL (20220215)')

In [ ]:
# ── Pre-load pre-fire scene NBR ───────────────────────────────────────
# For dNBR pairs, we need the pre-fire NBR from the same tile and position.
# Estructura: {stem_pre: {pos_suffix: ndarray(256,256)}}

def get_pos_suffix(path):
    """Extrae '_rXXXXX_cXXXXX' del nombre del patch."""
    stem = Path(path).stem
    idx = stem.find('_r')
    return stem[idx:] if idx != -1 else ''

pre_nbr_cache = {}

print('Pre-loading NBR from reference scenes...')
for pre_stem in PRE_FIRE_STEMS:
    pre_nbr_cache[pre_stem] = {}
    for p in sorted(IMG_DIR.glob(f'{pre_stem}_r*.npy')):
        pos = get_pos_suffix(p)
        img = np.load(p, mmap_mode='r')
        pre_nbr_cache[pre_stem][pos] = img[NBR_IDX].astype(np.float32) / 10000.0
    print(f'  {pre_stem}: {len(pre_nbr_cache[pre_stem])} patches')

In [ ]:
# ── Generate masks_dnbr for all patches ─────────────────────────────────
all_patches = sorted(IMG_DIR.glob('*.npy'))

stats = {'pre_fire': 0, 'dnbr': 0, 'nbr_only': 0}
positive_total = 0
dnbr_values_all = []

for img_path in tqdm(all_patches, desc='Generando máscaras'):
    stem       = img_path.stem
    scene_stem = stem.split('_r')[0]   # part before '_rXXXXX'
    pos        = get_pos_suffix(img_path)
    out_path   = MASK_DNBR / img_path.name

    img   = np.load(img_path)
    nbr   = img[NBR_IDX].astype(np.float32) / 10000.0
    valid = img[MASK_IDX] == 1

    if scene_stem in PRE_FIRE_STEMS:
        burn_mask = np.zeros((256, 256), dtype=np.uint8)
        stats['pre_fire'] += 1

    elif scene_stem in DNBR_PAIRS:
        pre_stem = DNBR_PAIRS[scene_stem]
        if pos in pre_nbr_cache[pre_stem]:
            nbr_pre = pre_nbr_cache[pre_stem][pos]
            dnbr    = nbr_pre - nbr
            burn_mask = ((dnbr > DNBR_THRESHOLD) & valid).astype(np.uint8)
            dnbr_values_all.extend(dnbr[valid].tolist())
        else:
            burn_mask = np.zeros((256, 256), dtype=np.uint8)
        stats['dnbr'] += 1

    else:
        burn_mask = ((nbr < NBR_THRESHOLD) & valid).astype(np.uint8)
        stats['nbr_only'] += 1

    np.save(out_path, burn_mask)
    if burn_mask.max() > 0:
        positive_total += 1

print(f'\n{"─"*50}')
print(f'Patches generated: {len(all_patches)}')
print(f'  Pre-fuego (zeros)   : {stats["pre_fire"]}')
print(f'  dNBR verdadero      : {stats["dnbr"]}')
print(f'  NBR umbral          : {stats["nbr_only"]}')
print(f'\nPatches CON cicatriz: {positive_total}/{len(all_patches)} ({positive_total/len(all_patches)*100:.1f}%)')
print(f'Antes (FIRMS)       : 148/5687 (2.6%)')
print(f'Mejora              : ~{positive_total//148}× more positive examples')

In [ ]:
# ── Distribución de dNBR ──────────────────────────────────────────────────────
if dnbr_values_all:
    dnbr_arr = np.array(dnbr_values_all, dtype=np.float32)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(dnbr_arr, bins=80, range=(-0.5, 1.2),
                 color='steelblue', edgecolor='none', alpha=0.8)
    axes[0].axvline(DNBR_THRESHOLD, color='red', lw=1.5, linestyle='--',
                    label=f'umbral={DNBR_THRESHOLD}')
    axes[0].set_xlabel('dNBR')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('dNBR Distribution — tiles 21JVJ + 21JWK')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    labels = ['Reverdecimiento\n(<-0.10)', 'Sin cambio\n(-0.10–0.10)',
              'Baja sev.\n(0.10–0.27)', 'Mod. baja\n(0.27–0.44)',
              'Mod. alta\n(0.44–0.66)', 'Alta sev.\n(>0.66)']
    bins_sev = [-np.inf, -0.10, 0.10, 0.27, 0.44, 0.66, np.inf]
    counts = [((dnbr_arr >= bins_sev[i]) & (dnbr_arr < bins_sev[i+1])).sum()
              for i in range(len(bins_sev)-1)]
    colors = ['#4CAF50', '#8BC34A', '#FFEB3B', '#FF9800', '#F44336', '#B71C1C']
    bars = axes[1].bar(range(len(labels)), counts, color=colors, edgecolor='white')
    axes[1].set_xticks(range(len(labels)))
    axes[1].set_xticklabels(labels, fontsize=7.5)
    axes[1].set_title('USGS Burn Severity Classification')
    axes[1].set_ylabel('Pixels')
    axes[1].grid(axis='y', alpha=0.3)
    for bar, cnt in zip(bars, counts):
        if sum(counts) > 0:
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(counts)*0.01,
                         f'{cnt/sum(counts)*100:.1f}%', ha='center', fontsize=8)

    plt.tight_layout()
    out = Path('..') / 'results' / 'dnbr_distribution.png'
    out.parent.mkdir(exist_ok=True)
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

In [ ]:
# ── Verificación visual — 3 patches con cicatriz ──────────────────────────────
mask_files     = sorted(MASK_DNBR.glob('*.npy'))
positive_masks = [f for f in mask_files if np.load(f, mmap_mode='r').max() > 0]
print(f'Patches positivos disponibles: {len(positive_masks)}')

if len(positive_masks) >= 3:
    sample_masks = positive_masks[:3]
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))

    for row, mf in enumerate(sample_masks):
        mask  = np.load(mf)
        img   = np.load(IMG_DIR / mf.name)
        firms = (np.load(MASK_FIRMS / mf.name)
                 if (MASK_FIRMS / mf.name).exists() else np.zeros((256,256), dtype=np.uint8))

        def norm(x):
            vals = x[x>0]
            if not len(vals): return np.zeros_like(x, dtype=np.float32)
            lo, hi = np.percentile(vals, [2, 98])
            return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)

        rgb = np.dstack([norm(img[2].astype(np.float32)),
                         norm(img[1].astype(np.float32)),
                         norm(img[0].astype(np.float32))])

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(mf.stem[:30], fontsize=7)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
        axes[row, 1].set_title(f'dNBR mask ({mask.mean()*100:.1f}% quemado)', fontsize=9)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(firms, cmap='Oranges', vmin=0, vmax=1)
        axes[row, 2].set_title(f'FIRMS mask ({firms.mean()*100:.1f}% fuego activo)', fontsize=9)
        axes[row, 2].axis('off')

    plt.suptitle('Comparison: dNBR burn scar (new) vs active fire FIRMS (previous)', fontsize=11)
    plt.tight_layout()
    out = Path('..') / 'results' / 'dnbr_vs_firms_comparison.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')
else:
    print(f'Solo {len(positive_masks)} patches positivos.')